# 3. PPO with KL Penalty — RLHF의 RL 단계

> 책 Chapter 8 — *Policy Gradient methods, PPO*

RLHF의 마지막이자 가장 복잡한 단계. SFT 모델을 초기 정책으로, RM을 보상 함수로 써서 **인간 선호를 최대화하는 정책을 학습**한다.

## 빌드업 순서

이 노트북은 다음 8단계로 차근차근 쌓는다:

1. **Policy Gradient Theorem** — RL의 출발점
2. **REINFORCE** — Monte Carlo PG
3. **Baselines** — 분산 감소
4. **Advantage** — A = R - V
5. **GAE** — Generalized Advantage Estimation
6. **PPO Clipped Objective** — surrogate loss와 clipping
7. **KL Penalty** — reward hacking 방지
8. **Full RLHF Loop** — rollout → score → update

각 단계는 *이전 단계의 한계*를 해결하면서 자연스럽게 나온다.

## 1. Policy Gradient Theorem — 어디서 출발하는가

목표:

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\bigl[R(\tau)\bigr]
$$

이를 직접 미분할 수 있나? 문제: $\tau$의 분포 자체가 $\theta$에 의존. 일반적인 미분 안 됨.

### Log-derivative trick

$$
\nabla_\theta P(\tau; \theta) = P(\tau; \theta) \nabla_\theta \log P(\tau; \theta)
$$

이를 활용:

$$
\nabla_\theta J(\theta)
= \nabla_\theta \int P(\tau; \theta) R(\tau)\, d\tau
= \int \nabla_\theta P(\tau; \theta) R(\tau)\, d\tau
= \int P(\tau; \theta) \nabla_\theta \log P(\tau; \theta) R(\tau)\, d\tau
$$

$$
= \mathbb{E}_{\tau \sim \pi_\theta}\bigl[\nabla_\theta \log P(\tau; \theta)\, R(\tau)\bigr]
$$

### Trajectory의 log-prob

$$
\log P(\tau; \theta) = \sum_{t} \log \pi_\theta(a_t \mid s_t) + \underbrace{\sum_t \log p(s_{t+1} \mid s_t, a_t)}_{\theta\text{와 무관, gradient 0}}
$$

따라서:

$$
\boxed{
\nabla_\theta J(\theta)
= \mathbb{E}\bigl[\sum_{t} \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau)\bigr]
}
$$

이게 **Policy Gradient Theorem**. RLHF에서:
- $s_t$ = 지금까지의 토큰 시퀀스
- $a_t$ = 다음 토큰
- $R(\tau)$ = reward model 점수 (응답 끝에 한 번)

## 2. REINFORCE — Monte Carlo Policy Gradient

PG theorem을 그대로 알고리즘으로 옮기면:

```
for iteration:
    1. π_θ로 trajectory τ 한 개 (또는 batch) 샘플
    2. R(τ) 계산
    3. ∇θ = Σ_t ∇θ log π_θ(a_t | s_t) · R(τ)
    4. θ ← θ + lr · ∇θ
```

### 문제점

1. **분산 폭발**: R이 sequence-level이라 noise 큼
2. **신용 할당 (credit assignment) 불명확**: 첫 토큰의 gradient도 마지막 reward에 영향받음
3. **샘플 비효율**: 매번 새 rollout 필요

In [ ]:
# REINFORCE의 가장 단순한 형태를 PyTorch로
import torch
import torch.nn.functional as F


def reinforce_loss(
    log_probs: torch.Tensor,   # (B, T) — 각 위치의 log π_θ(a_t | s_t)
    rewards: torch.Tensor,     # (B,)   — sequence-level R(τ)
) -> torch.Tensor:
    """REINFORCE: -Σ_t log π_θ(a_t | s_t) · R(τ)
    (gradient ascent를 위해 부호 뒤집어서 minimize 형태로 반환)
    """
    # 시퀀스 별로 step 합 → reward 곱 → 배치 평균
    sum_log_probs = log_probs.sum(dim=-1)                  # (B,)
    loss = -(sum_log_probs * rewards).mean()
    return loss


# 시연
torch.manual_seed(0)
log_probs = torch.randn(4, 10)         # 4 시퀀스, 10 토큰
rewards   = torch.tensor([1.0, -0.5, 0.2, 2.0])
print(f"REINFORCE loss: {reinforce_loss(log_probs, rewards).item():.4f}")

## 3. Baselines — 분산 줄이기

핵심 통찰: **모든 trajectory의 reward에서 *상수 b*를 빼도 gradient의 기댓값은 변하지 않는다.**

증명:

$$
\mathbb{E}\bigl[\nabla \log \pi_\theta(a \mid s) \cdot b\bigr]
= b \cdot \mathbb{E}\bigl[\nabla \log \pi_\theta(a \mid s)\bigr]
= b \cdot \nabla \mathbb{E}_\pi[1]
= 0
$$

(score function의 기댓값은 0이므로)

→ 따라서 다음으로 대체 가능:

$$
\nabla_\theta J \approx \mathbb{E}\Bigl[\sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot \bigl(R(\tau) - b\bigr)\Bigr]
$$

좋은 baseline 선택은 분산을 크게 줄임. 자주 쓰는 두 가지:
- **평균 baseline**: $b = \bar{R}$ (배치의 reward 평균)
- **State-value baseline**: $b = V(s_t)$ — *state-conditional*이라 더 정교

## 4. Advantage Function — A = R - V

Baseline을 state-dependent로 진화시킨 것:

$$
A(s_t, a_t) = R_t - V(s_t)
$$

- $V(s_t)$: state value — 현 상태에서 기대되는 미래 reward
- $A$: advantage — 그 state에서 그 action이 *평균보다 얼마나 좋았는가*

### 직관

$A > 0$: 이 action은 평균보다 좋았다 → 확률 증가시키자
$A < 0$: 평균보다 나빴다 → 확률 감소시키자

Reward 자체보다 advantage가 *상대적* 신호라 학습 안정성 ↑.

## 5. GAE — Generalized Advantage Estimation

Schulman et al. 2016. Advantage를 어떻게 estimate하나가 핵심 디자인 결정.

### 두 극단

- **Monte Carlo**: $A_t = R_t - V(s_t)$, 분산 ↑ but bias ↓
- **TD(0)**: $A_t = r_t + \gamma V(s_{t+1}) - V(s_t)$, 분산 ↓ but bias ↑

### GAE는 둘의 *지수 평균*

$$
A_t^{\mathrm{GAE}(\lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l},
\quad \text{where } \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

- $\lambda = 0$: 순수 TD(0) — 편향 큼, 분산 적음
- $\lambda = 1$: 순수 MC — 편향 0, 분산 큼
- 보통 $\lambda = 0.95 \sim 0.99$

### LLM RLHF에서

- $\gamma = 1.0$ (시퀀스가 짧고, 미래를 그대로 보고 싶음)
- $\lambda = 0.95$ (PPO 표준)
- $r_t = 0$ for $t < T$, $r_T = r_\phi(x, y)$ — RM이 마지막에 한 번만 신호 줌

In [ ]:
def compute_gae(
    rewards: torch.Tensor,    # (T,)
    values: torch.Tensor,     # (T + 1,) — bootstrap value 포함
    gamma: float = 1.0,
    lam: float = 0.95,
) -> tuple[torch.Tensor, torch.Tensor]:
    """GAE-λ advantage + return 계산.

    Args:
        rewards: 각 step의 reward. RLHF에선 마지막 step에만 non-zero.
        values:  각 step의 V(s_t). 마지막은 bootstrap (terminal에선 0).
        gamma:   discount.
        lam:     GAE λ.

    Returns:
        advantages: (T,)
        returns:    (T,) = advantages + values[:-1]  (value head 학습 target)
    """
    T = rewards.size(0)
    advantages = torch.zeros_like(rewards)
    last_gae = 0.0

    # 뒤에서부터 forward backward — Bellman recursion
    for t in reversed(range(T)):
        # TD residual
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        # GAE recursion: A_t = δ_t + γλ·A_{t+1}
        last_gae = delta + gamma * lam * last_gae
        advantages[t] = last_gae

    returns = advantages + values[:-1]
    return advantages, returns


# 시연: RLHF처럼 마지막에만 reward
T = 8
rewards = torch.zeros(T)
rewards[-1] = 1.5   # RM 점수 (마지막 step)

values = torch.tensor([0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 0.0])  # 마지막은 terminal bootstrap

adv, ret = compute_gae(rewards, values, gamma=1.0, lam=0.95)
print("Advantages:", adv.numpy().round(3))
print("Returns   :", ret.numpy().round(3))
# 관찰: advantage가 마지막 step부터 앞으로 전파됨

## 6. PPO Clipped Objective — 이 알고리즘의 정수

이전까지의 PG 알고리즘들의 큰 문제: **한 step에서 정책을 너무 크게 업데이트하면 collapse**.

PPO (Schulman et al. 2017)의 해법: **importance ratio를 clip해서 큰 변화를 막는다.**

### Importance Ratio

새 정책 $\pi_\theta$로 old 정책 $\pi_{\theta_\mathrm{old}}$의 데이터를 재사용하려면 importance sampling:

$$
r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\mathrm{old}}(a_t \mid s_t)}
$$

직관: 이 비율이 1이면 두 정책이 같은 확률로 행동. >1이면 새 정책이 이 action을 더 선호. <1이면 덜.

### Surrogate Objective

가장 단순한 형태:

$$
L^\mathrm{CPI}(\theta) = \mathbb{E}_t[r_t(\theta) \cdot A_t]
$$

(CPI = Conservative Policy Iteration). 문제: $r_t$가 너무 커지면 한 번에 정책이 폭주.

### Clipped 형태 (PPO의 핵심)

$$
\boxed{
L^\mathrm{CLIP}(\theta) = \mathbb{E}_t\Bigl[
\min\bigl(
r_t(\theta) \cdot A_t,\;
\mathrm{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \cdot A_t
\bigr)
\Bigr]
}
$$

### 이게 왜 작동하나? — 케이스 분석

**Case 1: A_t > 0** (이 action이 평균보다 좋음, 확률 ↑ 원함)
- $r_t$가 $1 + \epsilon$을 넘으면 clip → gradient = 0
- 즉 "이미 충분히 올렸다"고 보고 멈춤

**Case 2: A_t < 0** (이 action이 나쁨, 확률 ↓ 원함)
- $r_t$가 $1 - \epsilon$ 아래로 떨어지면 clip → gradient = 0
- 즉 "이미 충분히 낮췄다"고 보고 멈춤

`min()`은 "보수적인 쪽으로" 선택하게 만듦 — 정책이 너무 큰 step을 못 가도록.

보통 $\epsilon = 0.2$.

In [ ]:
def ppo_clipped_loss(
    log_probs_new: torch.Tensor,    # (B, T) — π_θ
    log_probs_old: torch.Tensor,    # (B, T) — π_{θ_old}, no_grad로 계산
    advantages: torch.Tensor,        # (B, T)
    clip_eps: float = 0.2,
    mask: torch.Tensor | None = None,  # (B, T) — valid token 1, padding 0
) -> dict[str, torch.Tensor]:
    """PPO clipped surrogate objective.

    Returns:
        dict with:
          loss      — minimize 형태 (즉 -L^CLIP)
          clip_frac — clip이 활성화된 토큰 비율 (모니터링)
          ratio_mean
          approx_kl — k1 KL estimate
    """
    # Importance ratio
    log_ratio = log_probs_new - log_probs_old        # log r_t
    ratio = log_ratio.exp()                           # r_t

    # 두 surrogate
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages

    # PPO는 보수적인 쪽 = min. ascent를 위해 부호 뒤집어 loss로 사용.
    clipped_advantage = torch.min(surr1, surr2)
    if mask is not None:
        # padding 무시
        loss = -(clipped_advantage * mask).sum() / mask.sum().clamp(min=1)
    else:
        loss = -clipped_advantage.mean()

    # 모니터링 지표
    clip_frac = ((ratio - 1.0).abs() > clip_eps).float()
    approx_kl = (log_ratio.exp() - 1.0 - log_ratio)   # k3 estimator
    if mask is not None:
        clip_frac = (clip_frac * mask).sum() / mask.sum().clamp(min=1)
        approx_kl = (approx_kl * mask).sum() / mask.sum().clamp(min=1)
    else:
        clip_frac = clip_frac.mean()
        approx_kl = approx_kl.mean()

    return {
        "loss": loss,
        "clip_frac": clip_frac,
        "ratio_mean": ratio.mean(),
        "approx_kl": approx_kl,
    }


# 시연
torch.manual_seed(0)
log_p_new = torch.randn(2, 5)
log_p_old = log_p_new - 0.1   # 약간 차이 (작은 update)
adv       = torch.randn(2, 5)
info = ppo_clipped_loss(log_p_new, log_p_old, adv, clip_eps=0.2)
for k, v in info.items():
    print(f"{k:15s} {v.item():.4f}")

## 7. Value Loss — Critic 학습

PPO에는 정책뿐 아니라 **value function $V_\psi$도 동시 학습**. Critic이라고도 부른다.

### Loss

단순 회귀 (MSE):

$$
L^V(\psi) = \mathbb{E}_t\bigl[(V_\psi(s_t) - R_t^\mathrm{target})^2\bigr]
$$

$R_t^\mathrm{target}$은 GAE에서 계산한 return.

### Clipped value loss (선택적)

PPO 원논문에 나오는 변형. value도 한 번에 크게 변하지 못하게:

$$
L^V_\mathrm{clipped} = \max\Bigl(
(V_\psi - R^\mathrm{target})^2,\;
(\mathrm{clip}(V_\psi, V_\mathrm{old} - \epsilon_v, V_\mathrm{old} + \epsilon_v) - R^\mathrm{target})^2
\Bigr)
$$

In [ ]:
def value_loss(
    values_new: torch.Tensor,     # (B, T) — V_ψ(s_t), 새 critic
    values_old: torch.Tensor,     # (B, T) — old critic (rollout 시)
    returns: torch.Tensor,         # (B, T) — GAE에서 계산한 target
    clip_eps: float = 0.2,
    mask: torch.Tensor | None = None,
) -> torch.Tensor:
    """Clipped value loss (PPO 표준)."""
    # 단순 MSE
    v_loss_unclipped = (values_new - returns).pow(2)

    # Clipped MSE
    values_clipped = values_old + torch.clamp(values_new - values_old, -clip_eps, clip_eps)
    v_loss_clipped = (values_clipped - returns).pow(2)

    v_loss = 0.5 * torch.max(v_loss_unclipped, v_loss_clipped)

    if mask is not None:
        return (v_loss * mask).sum() / mask.sum().clamp(min=1)
    return v_loss.mean()

## 8. KL Penalty — Reward Hacking 차단

PPO clipping만으로는 정책이 SFT에서 멀어지는 걸 *지속적*으로 막진 못함 (clip은 한 step 단위). RLHF에선 **추가로 KL penalty**를 reward에 직접 더한다.

### Reward 수정

$$
\tilde{r}_t = r_t - \beta \cdot \log \frac{\pi_\theta(a_t \mid s_t)}{\pi_\mathrm{ref}(a_t \mid s_t)}
$$

- $r_t$: RM 점수 (마지막 step에만 non-zero)
- $\beta$: KL 강도. 보통 0.01~0.2
- $\pi_\mathrm{ref}$: SFT 모델 (고정)

이렇게 "shaped reward"를 advantage 계산에 사용. **정책이 ref에서 멀어질수록 페널티가 reward에서 차감되어 advantage가 줄어듦.**

In [ ]:
def apply_kl_penalty(
    log_probs_policy: torch.Tensor,   # (B, T) — π_θ
    log_probs_ref: torch.Tensor,      # (B, T) — π_ref, no_grad
    rewards: torch.Tensor,             # (B, T) — RM 점수 (마지막 step에만 non-zero)
    beta: float = 0.05,
) -> torch.Tensor:
    """Shaped reward = r_t - β · log(π_θ / π_ref)

    각 토큰 단위로 KL 페널티를 reward에서 차감.
    RM은 시퀀스 끝에 한 번만 신호를 주지만, KL은 매 토큰마다.
    """
    log_ratio = log_probs_policy - log_probs_ref           # (B, T)
    shaped_rewards = rewards - beta * log_ratio
    return shaped_rewards


# 시연: 마지막 step에만 RM reward
B, T = 2, 6
rewards = torch.zeros(B, T)
rewards[:, -1] = torch.tensor([1.5, 0.8])

log_p_theta = torch.randn(B, T) * 0.1   # ref에서 약간만 멀어짐
log_p_ref   = torch.zeros(B, T)

shaped = apply_kl_penalty(log_p_theta, log_p_ref, rewards, beta=0.05)
print("Original rewards:")
print(rewards)
print("\nShaped rewards (with KL penalty):")
print(shaped.round(decimals=3))

## 9. 풀 RLHF Loop — 모든 것을 합치기

이제 모든 조각이 모였다. 한 번의 PPO iteration의 흐름:

```
[ROLLOUT PHASE]
  1. 프롬프트 batch x 샘플
  2. π_θ로 응답 y 생성 (autoregressive sampling)
  3. RM에 (x, y) 넣어 reward r_φ(x, y) 계산 — 시퀀스 끝에 한 번
  4. π_ref forward로 log π_ref(y_t | x, y_<t) 계산
  5. V_ψ forward로 V(s_t) 계산
  6. KL penalty 적용: r̃_t = r_t - β·log(π_θ/π_ref)
  7. GAE로 advantage A_t, return R_t 계산

[OPTIMIZATION PHASE (N epochs over rollout buffer)]
  for epoch in num_epochs:
      for minibatch in shuffle(buffer):
          policy_loss = ppo_clipped_loss(log_p_new, log_p_old, A)
          v_loss = value_loss(V_new, V_old, R)
          entropy_bonus = -E[log π_θ]  # 탐험 장려
          loss = policy_loss + c1·v_loss - c2·entropy_bonus
          loss.backward(); optimizer.step()
```

In [ ]:
# 전체 RLHF loop을 의사 코드로 구현
# 실행은 안 되지만 흐름을 끝까지 따라가도록 자세히 작성

from dataclasses import dataclass


@dataclass
class RolloutBuffer:
    """한 번의 rollout 결과를 담는 컨테이너."""
    input_ids:       torch.Tensor   # (B, T) — prompt + response 결합
    attention_mask:  torch.Tensor   # (B, T)
    log_probs_old:   torch.Tensor   # (B, T) — rollout 시점의 π_θ
    log_probs_ref:   torch.Tensor   # (B, T) — π_ref (KL 계산용)
    values_old:      torch.Tensor   # (B, T+1) — V_ψ(s_t), bootstrap 포함
    rewards_raw:     torch.Tensor   # (B, T) — RM 점수, 마지막에만 non-zero
    advantages:      torch.Tensor   # (B, T) — GAE
    returns:         torch.Tensor   # (B, T)
    response_mask:   torch.Tensor   # (B, T) — 1: 응답 토큰, 0: prompt / padding


def rollout_phase(
    policy,           # 학습 중인 LM (forward는 logits + value 둘 다 반환)
    reference,        # SFT 모델 (frozen)
    reward_model,     # 학습된 RM (frozen)
    prompts: list[str],
    tokenizer,
    sampling_kwargs: dict,
    beta_kl: float,
    gamma: float = 1.0,
    lam: float = 0.95,
) -> RolloutBuffer:
    """Rollout phase — 응답 생성하고 모든 필요한 신호를 모은다.

    실제 코드에서는 매 단계가 batched. 여기서는 의사 코드.
    """
    # 1) 토크나이즈
    prompt_ids = tokenizer(prompts, padding=True, return_tensors="pt").input_ids   # (B, P)

    # 2) π_θ로 응답 generate
    with torch.no_grad():
        response_ids = policy.generate(
            input_ids=prompt_ids,
            **sampling_kwargs,
        )   # (B, P + R)
        full_ids = response_ids   # 이미 prompt 포함

    # 3) Forward through π_θ — log probs + values 추출
    with torch.no_grad():
        out_policy = policy(input_ids=full_ids, return_value=True)
        logits_policy = out_policy["logits"]      # (B, T, V)
        values        = out_policy["values"]      # (B, T)

    # 4) Forward through π_ref — log probs (gradient 없음)
    with torch.no_grad():
        out_ref = reference(input_ids=full_ids).logits   # (B, T, V)

    # 5) Forward through reward model — 시퀀스당 scalar
    with torch.no_grad():
        rewards_scalar = reward_model(input_ids=full_ids)  # (B,)

    # 6) Per-token log probs 추출
    log_probs_policy = _gather_log_probs(logits_policy, full_ids)   # (B, T)
    log_probs_ref    = _gather_log_probs(out_ref,        full_ids)   # (B, T)

    # 7) Reward를 token-level로 펼치기 (마지막 응답 토큰에만 non-zero)
    rewards_token = torch.zeros_like(log_probs_policy)
    response_mask = _build_response_mask(full_ids, prompt_lengths=...)   # (B, T)
    last_idx = response_mask.sum(-1).long() - 1     # 마지막 응답 토큰 위치
    rewards_token[torch.arange(len(rewards_scalar)), last_idx] = rewards_scalar

    # 8) KL penalty 적용
    rewards_shaped = apply_kl_penalty(
        log_probs_policy, log_probs_ref, rewards_token, beta=beta_kl
    )

    # 9) GAE — 시퀀스마다
    advantages = torch.zeros_like(rewards_shaped)
    returns    = torch.zeros_like(rewards_shaped)
    for b in range(full_ids.size(0)):
        # bootstrap value 추가 (T+1)
        bootstrap = torch.cat([values[b], torch.zeros(1)])
        adv_b, ret_b = compute_gae(rewards_shaped[b], bootstrap, gamma=gamma, lam=lam)
        advantages[b] = adv_b
        returns[b]    = ret_b

    return RolloutBuffer(
        input_ids=full_ids,
        attention_mask=torch.ones_like(full_ids),
        log_probs_old=log_probs_policy.detach(),
        log_probs_ref=log_probs_ref.detach(),
        values_old=values.detach(),
        rewards_raw=rewards_token,
        advantages=advantages.detach(),
        returns=returns.detach(),
        response_mask=response_mask,
    )


def _gather_log_probs(logits: torch.Tensor, target_ids: torch.Tensor) -> torch.Tensor:
    """logits (B, T, V) + target_ids (B, T) → log P(target_t | ...) shape (B, T)"""
    log_probs = F.log_softmax(logits, dim=-1)
    return log_probs.gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)


def _build_response_mask(full_ids, prompt_lengths):
    """Placeholder — 실제로는 prompt_lengths를 알아야 함."""
    # 본 노트북은 시연이라 자세한 구현 생략
    return torch.ones_like(full_ids)


print("Rollout phase 정의 완료")

In [ ]:
def optimization_phase(
    policy,
    buffer: RolloutBuffer,
    optimizer,
    num_epochs: int = 4,
    minibatch_size: int = 16,
    clip_eps: float = 0.2,
    vf_coef: float = 0.1,
    ent_coef: float = 0.01,
    max_grad_norm: float = 1.0,
) -> dict[str, float]:
    """Optimization phase — buffer를 여러 epoch 돌면서 PPO update.

    PPO 핵심: 같은 데이터로 *N epoch* 재사용 (importance ratio로 보정).
    """
    metrics = {}
    B = buffer.input_ids.size(0)

    for epoch in range(num_epochs):
        # buffer 셔플
        perm = torch.randperm(B)

        for start in range(0, B, minibatch_size):
            idx = perm[start : start + minibatch_size]
            mb_input  = buffer.input_ids[idx]
            mb_attn   = buffer.attention_mask[idx]
            mb_lp_old = buffer.log_probs_old[idx]
            mb_v_old  = buffer.values_old[idx]
            mb_adv    = buffer.advantages[idx]
            mb_ret    = buffer.returns[idx]
            mb_mask   = buffer.response_mask[idx]

            # Advantage normalize (per minibatch, very common trick)
            mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

            # 새 forward — 현재 정책으로 log_probs와 value 다시 계산
            out = policy(input_ids=mb_input, attention_mask=mb_attn, return_value=True)
            logits_new = out["logits"]
            values_new = out["values"]

            log_p_new = _gather_log_probs(logits_new, mb_input)

            # 1) Policy loss (clipped surrogate)
            p_info = ppo_clipped_loss(
                log_p_new, mb_lp_old, mb_adv, clip_eps=clip_eps, mask=mb_mask,
            )
            p_loss = p_info["loss"]

            # 2) Value loss (clipped MSE)
            v_loss = value_loss(values_new, mb_v_old, mb_ret, clip_eps=clip_eps, mask=mb_mask)

            # 3) Entropy bonus — 탐험 유지
            #    H(π) ≈ -E[log π]. policy가 너무 빨리 deterministic으로 가는 걸 막음.
            probs = F.softmax(logits_new, dim=-1)
            log_probs_full = F.log_softmax(logits_new, dim=-1)
            entropy = -(probs * log_probs_full).sum(dim=-1)             # (B, T)
            entropy = (entropy * mb_mask).sum() / mb_mask.sum().clamp(min=1)

            # 4) 총 loss
            loss = p_loss + vf_coef * v_loss - ent_coef * entropy

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), max_grad_norm)
            optimizer.step()

            # 모니터링
            metrics["policy_loss"]   = p_loss.item()
            metrics["value_loss"]    = v_loss.item()
            metrics["entropy"]       = entropy.item()
            metrics["clip_frac"]     = p_info["clip_frac"].item()
            metrics["approx_kl"]     = p_info["approx_kl"].item()

    return metrics


print("Optimization phase 정의 완료")

In [ ]:
# 전체 RLHF training loop — 위 두 phase를 묶음

def rlhf_train(
    policy,
    reference,
    reward_model,
    prompt_dataset,
    tokenizer,
    num_iterations: int = 1000,
    rollout_batch_size: int = 64,
    num_epochs_per_rollout: int = 4,
    minibatch_size: int = 16,
    beta_kl: float = 0.05,
    learning_rate: float = 1e-6,
):
    """RLHF 책 Algorithm 8.x — 풀 학습 루프."""

    optimizer = torch.optim.AdamW(policy.parameters(), lr=learning_rate)

    sampling_kwargs = dict(
        max_new_tokens=256,
        do_sample=True,
        temperature=1.0,
        top_p=1.0,         # PPO에서는 보통 truncation 적게
    )

    for iteration in range(num_iterations):
        # 1) Prompt sampling
        prompts = prompt_dataset.sample(rollout_batch_size)

        # 2) Rollout
        buffer = rollout_phase(
            policy, reference, reward_model,
            prompts, tokenizer,
            sampling_kwargs=sampling_kwargs,
            beta_kl=beta_kl,
        )

        # 3) Optimization
        metrics = optimization_phase(
            policy, buffer, optimizer,
            num_epochs=num_epochs_per_rollout,
            minibatch_size=minibatch_size,
        )

        # 4) Logging
        if iteration % 10 == 0:
            print(
                f"iter {iteration:5d} | "
                f"policy {metrics['policy_loss']:+.3f} | "
                f"value {metrics['value_loss']:.3f} | "
                f"entropy {metrics['entropy']:.3f} | "
                f"clip {metrics['clip_frac']:.2f} | "
                f"approx_kl {metrics['approx_kl']:.4f}"
            )


print("RLHF training loop 정의 완료")

## 10. 학습 중 모니터링해야 할 지표들

PPO는 unstable하기로 악명 높음. 학습 중 반드시 봐야 할 신호:

| 지표 | 의미 | 위험 신호 |
|------|------|---------|
| `policy_loss` | clipped surrogate (작아져야 함) | 발산 |
| `value_loss` | critic MSE | 끝없이 큼 → critic capacity 부족 |
| `entropy` | 정책 entropy | 0으로 빨리 수렴 → mode collapse |
| `clip_frac` | clip 활성화 비율 | >0.3 → step 너무 큼, lr ↓ |
| `approx_kl` | π_θ ↔ π_old KL | >0.05 per step → early stop |
| **`reward_mean`** | 평균 reward (가장 중요) | 안 올라감 → 알고리즘 망가짐 |
| **`kl_to_ref`** | π_θ ↔ π_ref KL | 너무 큼 → reward hacking 시작 |

## 11. PPO의 실패 모드 — 잘 안 되는 경우

PPO는 잘 안 됨. 자주 보이는 실패:

1. **Reward hacking**
   - 증상: reward는 올라가는데 응답 품질은 떨어짐
   - 원인: RM이 cheap signal (길이, 키워드 등)에 너무 가중
   - 해결: β ↑ (KL 강화), RM 재학습, eval 다양화

2. **Mode collapse**
   - 증상: entropy → 0, 같은 응답만 생성
   - 원인: entropy bonus 너무 작음, clip이 너무 큼
   - 해결: ent_coef ↑, lr ↓

3. **Critic 발산**
   - 증상: value_loss 폭주
   - 원인: reward scale이 너무 큼, critic learning rate 너무 큼
   - 해결: reward normalize, separate critic lr

4. **빠른 KL 발산**
   - 증상: approx_kl > 0.1 in early steps
   - 원인: 첫 정책이 ref에서 너무 멀리 갈 수 있음
   - 해결: lr ↓, clip_eps ↓

→ **이런 이유로 DPO가 인기**. DPO는 RM 없이 supervised loss로 직접 학습.

## 12. 다시 큰 그림 — RLHF가 했던 일

이 4개 노트북 시리즈를 통해 우리가 본 것:

```
Pre-trained LM
   ↓ SFT (Notebook 1)
π_SFT (Kybalion 같은 instruction-tuned 모델)
   ↓ Reward Modeling (Notebook 2)
r_φ (인간 선호의 amortization)
   ↓ PPO with KL penalty (Notebook 3)
π_RLHF (정렬된 모델)
```

각 단계가 *수학적으로 왜 그렇게 생겼는지* 노트북별로 유도/구현했다:

- **SFT**: MLE → cross-entropy + masking
- **RM**: Bradley-Terry → sigmoid + log-likelihood
- **PPO**: PG theorem → REINFORCE → baseline → advantage → GAE → clipped surrogate → KL-shaped reward

이게 ChatGPT, Claude, Llama-Chat, Mistral-Instruct 같은 모든 RLHF 모델 뒤의 골격이다.

## 다음 단계 — 책의 어디로?

- **§7 Direct Alignment (DPO)**: RM 없이 한 번에. 우리 다른 노트북 `Kybalion_DPO.ipynb` 참고
- **§9 GRPO**: critic 없이 group-relative advantage (DeepSeek-R1)
- **§10 Constitutional AI**: human preference 대신 model-generated critique
- **§11 RLAIF**: AI feedback로 RM 학습
- **§12 Synthetic data**: rollout 자체를 합성으로

각 변형은 위에서 다룬 *어떤 부분을 제거하거나 대체*하는 방식으로 작동. 핵심 구조를 이해하면 모두 같은 골격의 변형으로 보임.